# 15장 · 통제의 기술과 한계

이 노트북은 구글 **Colab**에서 바로 실행됩니다. 설치는 없고, 구글 계정만 있으면 됩니다.

**읽는 법.** 지금 읽고 있는 이런 글은 설명이라 실행할 것이 없습니다. 코드는 **왼쪽에 `[ ]` 와 ▶ 단추가 붙은 상자**이고, 그 ▶ 또는 `Shift`+`Enter` 로 실행합니다.

**순서.** 번호 차례대로 끝까지 갑니다. **1 준비**부터 시작해 마지막 번호까지 위에서 아래로 내려가면 됩니다. ⚠ 가운데부터 누르면 앞에서 만든 것이 없어 오류가 납니다.

📖 본문 학습 페이지: [15장 · 통제의 기술과 한계](https://grow.minds.kr/textbooks/css-methods/causal/book/ch15-통제의-기술과-한계.html)

## 1. 준비

아래 상자 **둘**을 차례로 실행하세요. 둘 다 해야 그다음이 돌아갑니다.

1. 첫째 = 자료와 코드를 내려받습니다. 몇 초 걸리고 「경고」 문구가 떠도 정상입니다. 상자 아래에 `/content/css-methods-causal-code` 가 찍히면 성공입니다.
2. 둘째 = 도구를 불러옵니다. **준비 끝**이라는 줄이 찍히면 됩니다.

⚠ 한참 뒤에 돌아와 `pd`·`exp`가 없다거나 파일이 없다는 오류가 나면 코랩이 연결을 끊은 것입니다. 자료는 내 컴퓨터가 아니라 코랩의 컴퓨터에 있었으므로 함께 사라졌습니다. **이 두 상자를 위에서부터 다시** 누르면 됩니다(첫째 상자는 여러 번 눌러도 됩니다).

> **어느 것이 상자인가.** 왼쪽에 `[ ]` 표시와 **▶** 단추가 붙은 것이 코드 상자입니다. 글씨에 색이 섞여 있습니다. 지금 읽고 있는 이런 글에는 `[ ]` 도 ▶ 도 없습니다.

In [ ]:
# 이 책의 데이터·코드를 코랩으로 내려받습니다(수 초). 다시 눌러도 됩니다.
%cd /content
![ -d css-methods-causal-code ] || git clone -q https://github.com/dataminds/css-methods-causal-code.git
%cd /content/css-methods-causal-code

In [ ]:
import pandas as pd, numpy as np
from scipy import stats

def load(name, clean=True):
    df = pd.read_csv(f"data/journey_{name}.csv")
    return df[df.attn_1 == 1] if clean and "attn_1" in df else df

def ols(y, X):                      # 절편 포함 최소제곱 → (계수, 표준오차, p, R^2)
    y = np.asarray(y, float)
    X1 = np.column_stack([np.ones(len(y))] + [np.asarray(x, float) for x in X])
    b, *_ = np.linalg.lstsq(X1, y, rcond=None)
    resid = y - X1 @ b
    n, k = X1.shape
    se = np.sqrt(np.diag(resid @ resid / (n - k) * np.linalg.inv(X1.T @ X1)))
    p = 2 * stats.t.sf(np.abs(b / se), n - k)
    r2 = 1 - (resid @ resid) / ((y - y.mean()) @ (y - y.mean()))
    return b, se, p, r2

def cohen_d(a, b):
    sp = np.sqrt(((len(a)-1)*a.std(ddof=1)**2 + (len(b)-1)*b.std(ddof=1)**2) / (len(a)+len(b)-2))
    return (a.mean() - b.mean()) / sp

def cronbach(items):
    items = np.asarray(items, float); k = items.shape[1]
    return k/(k-1) * (1 - items.var(axis=0, ddof=1).sum() / items.sum(axis=1).var(ddof=1))

print("준비 끝. 데이터와 도우미 함수를 불러왔습니다.")


💡 **방금 둘째 상자에 코드가 길게 펼쳐졌는데, 지금 이해하지 않아도 됩니다.**

이 책 전체가 쓰는 **도우미 넷**을 미리 만들어 두는 곳이라 그렇습니다. 지금은 이름과 하는 일만 훑고 넘어가세요. 안에 든 식은 필요한 장에서 하나씩 만납니다.

| 이름 | 하는 일 | 처음 쓰는 곳 |
|---|---|---|
| `load` | 자료 한 벌을 불러오고 주의 점검 실패자를 걸러 낸다 | 2장 |
| `cronbach` | 척도 신뢰도(알파) | 4장 |
| `cohen_d` | 두 집단 차이의 효과크기 | 13장 |
| `ols` | 회귀의 계수·표준오차·p·R² | 14장 |

⚠ `np.linalg.lstsq` 처럼 낯선 이름이 보여도 괜찮습니다. **그것이 이 책이 「계산은 AI가」라고 말하는 곳입니다.**

⭐ **그리고 이 넷은 여러분이 검증할 대상이 아닙니다.** 이 책이 검증해 둔 도구입니다(배포 때마다 대조 배터리가 이 함수들의 출력을 본문 수치와 맞춰 봅니다). 여러분이 읽어야 할 코드는 **AI가 준 코드**이고, 그 훈련은 3장부터 시작합니다.

📖 이 넷을 다시 보고 싶으면 책 부록 E.1b 에 같은 코드가 있습니다.

## 2. 다중회귀 = 통계적 통제
저그마을(8장): 성별을 통제하자 복용량 계수가 +1.70에서 -1.69로 뒤집힙니다.

In [ ]:
def make_simpson(seed=73, n=400):
    rng = np.random.default_rng(seed); sex = rng.integers(0, 2, n)
    dose = np.clip(rng.normal(2 + 4*sex, 1.5), 0, 10)
    vigor = 55 + 20*sex - 1.5*dose + rng.normal(0, 4, n)
    return pd.DataFrame({"sex": sex, "dose": dose.round(1), "vigor": vigor.round(1)})
zg = make_simpson()
b,_,_,_ = ols(zg.vigor.values, [zg.dose.values]); print("복용량 단독:", round(b[1],2))       # +1.70
b,_,_,_ = ols(zg.vigor.values, [zg.dose.values, zg.sex.values])
print("성별 통제 후 복용량:", round(b[1],2), " 성별 계수:", round(b[2],2))                    # -1.69, +20.14

## 3. 과통제 함정(충돌 변수)
세대성(gen)을 통제하면 오히려 계수가 왜곡됩니다(0.98→0.86).

In [ ]:
svy = load("svy")
b,_,_,_ = ols(svy.mil.values, [svy.hjs.values]); print("mil ~ hjs:", round(b[1],2))            # 0.98
b,_,_,_ = ols(svy.mil.values, [svy.hjs.values, svy.gen.values]); print("+ gen:", round(b[1],2))  # 0.86

## 4. (책이 원리를 보이려고 돌린 코드) 과소적합 · 적합 · 과적합 (책 §15.1 끝 [선택] 소절)
참 관계가 완만한 곡선인 세계를 짓고(씨앗 73), 30명에게 1 · 2 · 12차 모형을 맞춘 뒤 같은 세계에서 새로 뽑은 1,000명에 적용합니다. `np.polyfit(x, y, 차수)`는 차수만큼 구부러진 다항식을 최소제곱으로 맞추고(1차 = 곧은 선, 2차 = 포물선), `np.polyval`은 맞춘 식에 값을 넣어 적합값을 냅니다. 둘째 반복은 같은 설계로 세계를 300번 새로 지어, 새 자료의 R²가 0보다 낮았던 세계의 비율과 중앙값을 셉니다. 12차 모형은 numpy 가 조건이 나쁘다는 경고(RankWarning)를 낼 수 있습니다.

In [ ]:
g = np.random.default_rng(73)
참 = lambda v: 1 + 1.6 * v - 0.16 * v ** 2                         # 참 관계 = 완만한 곡선
x30 = g.uniform(0, 10, 30);   y30 = 참(x30) + g.normal(0, 1, 30)     # 지금 자료 30명
x1k = g.uniform(0, 10, 1000); y1k = 참(x1k) + g.normal(0, 1, 1000)   # 새 자료 1,000명
R2 = lambda yy, f: 1 - ((yy - f) ** 2).sum() / ((yy - yy.mean()) ** 2).sum()
for 차수 in (1, 2, 12):
    c = np.polyfit(x30, y30, 차수)
    print(차수, round(R2(y30, np.polyval(c, x30)), 2), round(R2(y1k, np.polyval(c, x1k)), 2))
print(round(R2(y1k, 참(x1k)), 2))
# 1 0.07 -0.07
# 2 0.71 0.53
# 12 0.85 0.21
# 0.6

g = np.random.default_rng(73)
새R2 = {1: [], 2: [], 12: []}
for _ in range(300):                                                # 세계를 300번 새로 짓는다
    x30 = g.uniform(0, 10, 30);   y30 = 참(x30) + g.normal(0, 1, 30)
    x1k = g.uniform(0, 10, 1000); y1k = 참(x1k) + g.normal(0, 1, 1000)
    for 차수 in 새R2:
        c = np.polyfit(x30, y30, 차수)
        새R2[차수].append(R2(y1k, np.polyval(c, x1k)))
for 차수, v in 새R2.items():
    v = np.array(v)
    print(차수, round(float(np.mean(v < 0)), 2), round(float(np.median(v)), 2))
# 1 0.97 -0.05
# 2 0.0 0.55
# 12 0.72 -2.54

## 5. (책이 원리를 보이려고 돌린 코드) 원인끼리 무관한 세계 (책 §15.1 「원인끼리 무관하면: 합산모형」 · 코드 12)
저그마을 생성기에서 성별 → 복용량 다리만 끊은 `make_free`를 만들고, 묶인 마을(위 2 의 `make_simpson`)과 끊은 마을에서 세 모형(복용량만 · 성별만 · 둘 함께)을 맞춥니다. 마을마다 한 줄에 성별과 복용량의 상관, 복용량 계수 둘(복용량만 · 성별 함께), 그 *SE* 둘, 성별 계수 둘(성별만 · 복용량 함께), *R*² 넷(복용량만 · 성별만 · 그 합 · 둘 함께)을 차례로 찍습니다. 끊은 마을에서는 통제해도 복용량 계수가 거의 그대로이고 *SE* 만 줄어듭니다.

In [ ]:
def make_free(seed=73, n=400):                     # 성별 → 복용량 다리만 끊은 저그마을
    rng = np.random.default_rng(seed)
    sex = rng.integers(0, 2, n)
    dose = np.clip(rng.normal(4, 2.5, n), 0, 10)   # 성별과 상관없이 뽑는다
    vigor = 55 + 20 * sex - 1.5 * dose + rng.normal(0, 4, n)
    return pd.DataFrame({"sex": sex, "dose": dose.round(1), "vigor": vigor.round(1)})

for name, z in (("묶임", make_simpson()), ("끊음", make_free())):
    bd, sd, _, r2d = ols(z.vigor, [z.dose])             # 복용량만
    bs, _, _, r2s = ols(z.vigor, [z.sex])               # 성별만
    bj, sj, _, r2j = ols(z.vigor, [z.dose, z.sex])      # 둘 함께
    print(name, round(np.corrcoef(z.sex, z.dose)[0, 1], 2),
          round(bd[1], 2), round(bj[1], 2), round(sd[1], 2), round(sj[1], 2),
          round(bs[1], 2), round(bj[2], 2),
          round(r2d, 3), round(r2s, 3), round(r2d + r2s, 3), round(r2j, 3))
# 묶임 0.81 1.7 -1.69 0.15 0.14 13.57 20.14 0.249 0.683 0.932 0.768
# 끊음 -0.02 -1.69 -1.62 0.23 0.09 19.54 19.4 0.123 0.763 0.886 0.875

## 6. 실습 7: 세계행복보고서 자료로 통제를 판정한다 (책 §15.8 실습 7)
⚠ 이 노트북에는 자료가 들어 있지 않습니다. 직접 내려받습니다.

1. 세계행복보고서(World Happiness Report) 누리집(https://worldhappiness.report)의 자료 쪽에서 나라별 자료 파일을 내려받습니다. 엑셀 파일이면 엑셀에서 열어 CSV 로 저장합니다.
2. 코랩 왼쪽의 폴더 아이콘을 눌러 그 CSV 를 올립니다(올린 파일은 연결이 끊기면 사라집니다).
3. 아래 상자의 `파일` 과 열 이름 셋을 내 파일에 맞게 고칩니다. 열 이름은 판마다 조금씩 다릅니다(예 = `Life Ladder` · `Log GDP per capita` · `Social support`). `print(df.columns.tolist())` 로 확인합니다.

⭐ 상자를 돌리기 **전에** 세 변수의 DAG 와 예측(GDP 가 혼란변수인가 · 사슬의 가운데인가 · 둘 다 아닌가)을 먼저 씁니다. 어느 판의 어느 해를 썼는지와 내려받은 날짜도 적어 둡니다. 파일이 없으면 상자는 안내만 찍고 끝납니다.

In [ ]:
import os
파일 = "whr.csv"                   # 올린 파일 이름으로 바꾼다
삶 = "Life Ladder"                 # 삶의 평가(사다리 점수)
지지 = "Social support"            # 사회적 지지
gdp = "Log GDP per capita"         # 1인당 GDP 의 로그
if not os.path.exists(파일):
    print("파일이 없습니다. 위 안내대로 내려받아 올린 뒤 파일 이름을 고치세요.")
else:
    df = pd.read_csv(파일)
    print(df.columns.tolist())
    # 한 해만 쓸 때 = 해 열 이름을 확인해 고른다. 예: df = df[df["year"] == 2023]
    w = df[[삶, 지지, gdp]].dropna()     # 세 열이 다 있는 행만
    print("나라 · 해 행 수:", len(w))
    b1, se1, _, r2_1 = ols(w[삶], [w[지지]])              # ② 사회적 지지만
    bg, _, _, r2_g = ols(w[삶], [w[gdp]])                # GDP 만
    b2, se2, _, r2_2 = ols(w[삶], [w[지지], w[gdp]])      # ③ 둘 함께
    print("사회적 지지 계수: 단독", round(b1[1], 3), "(SE", round(se1[1], 3), ")",
          "→ GDP 함께", round(b2[1], 3), "(SE", round(se2[1], 3), ")")
    print("사회적 지지와 GDP 의 상관:", round(np.corrcoef(w[지지], w[gdp])[0, 1], 2))
    print("R² 지지만", round(r2_1, 3), "· GDP 만", round(r2_g, 3),
          "· 합", round(r2_1 + r2_g, 3), "· 둘 함께", round(r2_2, 3))

## 7. 직접 바꿔 보기
위 셀의 숫자(씨앗 73, 표본 크기, 제외 기준 등)를 바꿔 다시 실행해 보세요. 결과가 어떻게 달라지나요?

> **검증 로그(부록 B)**: 무엇을 바꿨고, 무엇이 나왔고, 예상과 같았는지 한 문단으로 적어 두세요. 실행이 아니라 검증이 이 책의 핵심입니다.